# EDA — MVTec AD + HSS-IAD (Unified Dataset)

**Pré-requis :** exécuter `uv run python -m src.data.harmonize` pour générer les CSV.

**Sommaire :**
1. Chargement & vue d'ensemble
2. Distribution des images par catégorie et dataset
3. Distribution des résolutions d'images
4. Exemples visuels (normal vs anomal + masque)
5. Analyse des masques — ratio de pixels anomaux
6. Analyse de la diversité des canaux (RGB vs Grayscale)
7. Synthèse & recommandations

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Rendre la racine du projet importable quand le notebook est lancé
# depuis le dossier notebooks/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA, EDA, PATHS

warnings.filterwarnings("ignore")
sns.set_theme(style=EDA.sns_style, palette=EDA.sns_palette, font_scale=EDA.sns_font_scale)
plt.rcParams["figure.dpi"] = EDA.figure_dpi

df = pd.read_csv(PATHS.unified_csv)
df_res = pd.read_csv(PATHS.resolutions_csv)

print(f"Dataset unifié : {len(df):,} images | {df['category'].nunique()} catégories | {df['dataset'].nunique()} sources")
df.head()

## 1. Vue d'ensemble

In [ ]:
# Résumé global par dataset
overview = df.groupby("dataset").agg(
    images=("image_path", "count"),
    categories=("category", "nunique"),
    anomalies=("is_anomaly", "sum"),
    masks=("has_mask", "sum"),
).assign(
    normal=lambda x: x["images"] - x["anomalies"],
    anomaly_pct=lambda x: (x["anomalies"] / x["images"] * 100).round(1),
    mask_coverage_pct=lambda x: (x["masks"] / x["anomalies"] * 100).round(1),
)
print(overview.to_string())

print("\n--- Détail par catégorie et split ---")
detail = df.groupby(["dataset", "category", "split"]).agg(
    total=("image_path", "count"),
    anomalies=("is_anomaly", "sum"),
    masks=("has_mask", "sum"),
).reset_index()
detail["normal"] = detail["total"] - detail["anomalies"]
detail

## 2. Distribution par catégorie — volumes, normal/anomal et types de défauts

Vue consolidée par catégorie combinant trois dimensions :
- **Hauteur totale** = nombre d'images
- **Couleur** = dataset (bleu = MVTec, orange = HSS-IAD)
- **Opacité** = Normal (transparent) vs Anomal (plein)
- **Annotation** au-dessus de chaque barre = nombre de types de défauts distincts

Les catégories sont groupées par dataset puis triées par volume décroissant.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

# Ordre : MVTec d'abord (desc par volume), puis HSS-IAD (desc par volume)
totals = df.groupby(["category", "dataset"]).size().reset_index(name="total")
mvtec_cats = totals[totals["dataset"] == DATA.mvtec_name].sort_values("total", ascending=False)
hssiad_cats = totals[totals["dataset"] == DATA.hssiad_name].sort_values("total", ascending=False)
ordered = pd.concat([mvtec_cats, hssiad_cats]).reset_index(drop=True)
cat_order = ordered["category"].tolist()
cat_datasets = dict(zip(ordered["category"], ordered["dataset"]))

# Stats par catégorie
normal_counts = [int(((df["category"] == c) & (~df["is_anomaly"])).sum()) for c in cat_order]
anomal_counts = [int(((df["category"] == c) & (df["is_anomaly"])).sum()) for c in cat_order]
colors = [EDA.color_mvtec if cat_datasets[c] == DATA.mvtec_name else EDA.color_hssiad for c in cat_order]

# Nombre de types de défauts par catégorie
defect_types = df[df["is_anomaly"]].groupby("category")["label"].nunique().to_dict()

# Colormap bleu -> jaune pour encoder le nb de types (1 = bleu, 8 = jaune)
cmap_types = LinearSegmentedColormap.from_list("blue_yellow", ["#1f77b4", "#FFD700"])
types_min, types_max = 1, 8
norm_types = Normalize(vmin=types_min, vmax=types_max)

fig, ax = plt.subplots(figsize=(16, 7.5))
x = np.arange(len(cat_order))
width = 0.75

# Pile : Normal (transparent) en bas + Anomal (plein) au-dessus
ax.bar(x, normal_counts, width, color=colors, alpha=0.45, edgecolor="white", linewidth=0.3)
ax.bar(x, anomal_counts, width, bottom=normal_counts, color=colors, alpha=1.0,
       edgecolor="black", linewidth=0.4)

# Marqueur circulaire coloré au-dessus de chaque barre (gradient bleu->jaune selon nb de types)
max_total = max(n + a for n, a in zip(normal_counts, anomal_counts))
offset = max_total * 0.035
for i, cat in enumerate(cat_order):
    total = normal_counts[i] + anomal_counts[i]
    n_types = defect_types.get(cat, 0)
    marker_color = cmap_types(norm_types(n_types))
    ax.scatter(x[i], total + offset, s=360, color=marker_color,
               edgecolor="black", linewidth=0.8, zorder=3)
    ax.text(x[i], total + offset, str(n_types), ha="center", va="center",
            fontsize=9, fontweight="bold", color="black", zorder=4)

# Séparateur entre MVTec et HSS-IAD
n_mvtec = len(mvtec_cats)
ax.axvline(x=n_mvtec - 0.5, color="gray", linestyle="--", alpha=0.4, linewidth=1)
ax.text((n_mvtec - 1) / 2, max_total * 1.06, "MVTec AD", ha="center",
        fontsize=12, fontweight="bold", color=EDA.color_mvtec, alpha=0.8)
ax.text(n_mvtec + (len(hssiad_cats) - 1) / 2, max_total * 1.06, "HSS-IAD", ha="center",
        fontsize=12, fontweight="bold", color=EDA.color_hssiad, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(cat_order, rotation=45, ha="right")
ax.set_ylabel("Nombre d'images")
ax.set_xlabel("Catégorie")
ax.set_title("Distribution par catégorie — volumes, statut Normal/Anomal, nombre de types de défauts")

ax.set_ylim(0, max_total * 1.13)

# Légende principale : dataset × statut
legend_elements = [
    Patch(facecolor=EDA.color_mvtec, alpha=0.45, label="MVTec — Normal"),
    Patch(facecolor=EDA.color_mvtec, alpha=1.0, edgecolor="black", label="MVTec — Anomal"),
    Patch(facecolor=EDA.color_hssiad, alpha=0.45, label="HSS-IAD — Normal"),
    Patch(facecolor=EDA.color_hssiad, alpha=1.0, edgecolor="black", label="HSS-IAD — Anomal"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.95)

# Colorbar pour le gradient bleu->jaune (nb de types de défauts)
sm = ScalarMappable(norm=norm_types, cmap=cmap_types)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.015)
cbar.set_label("Nb de types de défauts distincts", fontsize=9)
cbar.set_ticks(range(types_min, types_max + 1))

plt.tight_layout()
plt.show()

# Récap texte
print("Récap :")
print(f"  MVTec   : {sum(1 for c in cat_order if cat_datasets[c] == DATA.mvtec_name)} catégories, "
      f"{sum(normal_counts[i] + anomal_counts[i] for i, c in enumerate(cat_order) if cat_datasets[c] == DATA.mvtec_name):,} images")
print(f"  HSS-IAD : {sum(1 for c in cat_order if cat_datasets[c] == DATA.hssiad_name)} catégories, "
      f"{sum(normal_counts[i] + anomal_counts[i] for i, c in enumerate(cat_order) if cat_datasets[c] == DATA.hssiad_name):,} images")
print(f"  Max types de défauts : {max(defect_types.values())} ({max(defect_types, key=defect_types.get)})")
print(f"  Catégories 1 seul type : {sum(1 for v in defect_types.values() if v == 1)}")


## 3. Distribution des résolutions d'images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Subplot 1 : scatter Largeur vs Hauteur ---
max_val = max(df_res["width"].max(), df_res["height"].max()) * 1.05
min_val = 0

# Fonds colorés de part et d'autre de la diagonale x=y
axes[0].fill_between([min_val, max_val], [min_val, min_val], [min_val, max_val],
                     color="#E8F4F8", alpha=0.7, zorder=0)  # Paysage (bas-droite)
axes[0].fill_between([min_val, max_val], [min_val, max_val], [max_val, max_val],
                     color="#FBE8E8", alpha=0.7, zorder=0)  # Portrait (haut-gauche)

# Diagonale x=y en pointillés
axes[0].plot([min_val, max_val], [min_val, max_val],
             linestyle="--", color="black", alpha=0.6, linewidth=1.2, label="x = y (carré)")

# Labels des zones
axes[0].text(max_val * 0.60, max_val * 0.30, "Paysage\n(w > h)",
             fontsize=10, style="italic", alpha=0.65, ha="center", va="center")
axes[0].text(max_val * 0.15, max_val * 0.75, "Portrait\n(h > w)",
             fontsize=10, style="italic", alpha=0.65, ha="center", va="center")

# Scatter : marker carré (s) si image carrée, rond (o) sinon
is_square = (df_res["width"] == df_res["height"])

for ds, color in [(DATA.mvtec_name, EDA.color_mvtec), (DATA.hssiad_name, EDA.color_hssiad)]:
    m_ds = df_res["dataset"] == ds
    # Carrées : marker carré
    m_sq = m_ds & is_square
    axes[0].scatter(df_res.loc[m_sq, "width"], df_res.loc[m_sq, "height"],
                    alpha=0.7, label=f"{ds} — carré", color=color, s=40,
                    marker="s", edgecolor="black", linewidth=0.4, zorder=3)
    # Non carrées : marker rond
    m_ns = m_ds & ~is_square
    axes[0].scatter(df_res.loc[m_ns, "width"], df_res.loc[m_ns, "height"],
                    alpha=0.55, label=f"{ds} — non carré", color=color, s=28,
                    marker="o", zorder=2)

axes[0].set_xlim(min_val, max_val)
axes[0].set_ylim(min_val, max_val)
axes[0].set_aspect("equal")
axes[0].set_xlabel("Largeur (px)")
axes[0].set_ylabel("Hauteur (px)")
axes[0].set_title("Résolutions : Largeur vs Hauteur")
axes[0].legend(fontsize=8, loc="upper right")

# --- Subplot 2 : top 10 résolutions ---
df_res["resolution"] = df_res["width"].astype(int).astype(str) + "x" + df_res["height"].astype(int).astype(str)
res_counts = df_res.groupby(["resolution", "dataset"]).size().unstack(fill_value=0)
res_counts = res_counts.loc[res_counts.sum(axis=1).nlargest(10).index]
res_counts.plot(kind="barh", stacked=True, ax=axes[1], color=[EDA.color_hssiad, EDA.color_mvtec])
axes[1].set_xlabel("Nombre d'images (échantillon)")
axes[1].set_title("Top 10 résolutions")
axes[1].legend(title="Dataset")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nRésolutions uniques par dataset :")
for ds in df_res["dataset"].unique():
    resolutions = df_res[df_res["dataset"] == ds]["resolution"].unique()
    print(f"  {ds}: {len(resolutions)} résolutions — exemples : {sorted(resolutions)[:5]}")

print(f"\nImages carrées : {int(is_square.sum())}/{len(df_res)} ({is_square.mean()*100:.1f}%)")
print(f"  par dataset : {df_res.groupby('dataset').apply(lambda g: (g['width'] == g['height']).mean()*100).round(1).to_dict()}")


## 4. Exemples visuels — Normal vs Anomal + Masque

Pour chaque dataset, on affiche quelques catégories avec : image normale, image anomale, masque associé.

In [ ]:
import re

import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgba

# --- Palette & style locaux pour la section 4 ----------------------------
_NORMAL_C = EDA.color_normal   # vert
_ANOMAL_C = EDA.color_anomal   # rouge
_MASK_C   = "#555555"          # gris foncé

# Colormap transparent -> rouge pour l'overlay de masque
_MASK_CMAP = LinearSegmentedColormap.from_list(
    "mask_overlay",
    [(0, (*to_rgba(_ANOMAL_C)[:3], 0.0)),
     (1, (*to_rgba(_ANOMAL_C)[:3], 0.85))],
)

# Pour HSS-IAD Casting : suffixe de vue _1_2, _1_3, _2_3 -> apparier les vues
_VIEW_RE = re.compile(r"_(\d+_\d+)$")


def _view_key(stem: str) -> str | None:
    m = _VIEW_RE.search(stem)
    return m.group(1) if m else None


def _style_panel(ax, color: str, title: str):
    """Applique un cadre coloré + un titre mis en forme à un sous-plot."""
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(2.2)
    ax.set_title(title, fontsize=11, color=color, fontweight="bold", pad=6)


def show_samples(df, dataset_name, n_categories=EDA.n_example_categories, seed=EDA.example_seed):
    """Affiche, pour n_categories aléatoires, un triplet :
    Normal | Anomal (+ overlay masque) | Masque seul.

    Pour HSS-IAD Casting, on apparie Normal et Anomal sur la même vue
    (suffixe `_1_2`, `_1_3`, `_2_3`).
    """
    rng = np.random.RandomState(seed)
    sub = df[df["dataset"] == dataset_name]
    categories = rng.choice(sub["category"].unique(),
                            size=min(n_categories, sub["category"].nunique()),
                            replace=False)

    fig, axes = plt.subplots(
        len(categories), 3,
        figsize=(12.5, 4.2 * len(categories)),
        gridspec_kw={"wspace": 0.08, "hspace": 0.35},
    )
    if len(categories) == 1:
        axes = axes[np.newaxis, :]

    fig.patch.set_facecolor("white")

    for i, cat in enumerate(categories):
        cat_df = sub[sub["category"] == cat]

        # 1) Anomal + masque
        anomal = cat_df[(cat_df["is_anomaly"] == True) & (cat_df["has_mask"] == True)]
        anom_row, view = None, None
        if len(anomal) > 0:
            anom_row = anomal.iloc[rng.randint(len(anomal))]
            view = _view_key(Path(anom_row["image_path"]).stem)

        # 2) Normal (même vue si possible)
        normal = cat_df[(cat_df["is_anomaly"] == False) & (cat_df["split"] == "test")]
        if len(normal) == 0:
            normal = cat_df[cat_df["is_anomaly"] == False]
        if view is not None:
            same_view = normal[normal["image_path"].apply(
                lambda p: _view_key(Path(p).stem) == view
            )]
            if len(same_view) > 0:
                normal = same_view

        view_str = f"  ·  vue {view}" if view else ""

        # --- Colonne 0 : Normal ---------------------------------------
        if len(normal) > 0:
            img = Image.open(PATHS.root / normal.iloc[rng.randint(len(normal))]["image_path"])
            axes[i, 0].imshow(img)
        _style_panel(axes[i, 0], _NORMAL_C, f"Normal{view_str}")

        # --- Colonne 1 : Anomal + overlay masque ----------------------
        if anom_row is not None:
            img = Image.open(PATHS.root / anom_row["image_path"])
            mask = np.array(Image.open(PATHS.root / anom_row["mask_path"]).convert("L"))
            axes[i, 1].imshow(img)
            axes[i, 1].imshow(mask, cmap=_MASK_CMAP, vmin=0, vmax=255)
            _style_panel(axes[i, 1], _ANOMAL_C, f"Anomal — {anom_row['label']}")
        else:
            _style_panel(axes[i, 1], _ANOMAL_C, "(pas d'anomalie)")

        # --- Colonne 2 : Masque seul ----------------------------------
        if anom_row is not None:
            mask = np.array(Image.open(PATHS.root / anom_row["mask_path"]).convert("L"))
            pct = (mask > 0).sum() / mask.size * 100
            axes[i, 2].imshow(mask, cmap="gray_r")
            _style_panel(axes[i, 2], _MASK_C, f"Masque  ·  {pct:.2f}% px")
        else:
            _style_panel(axes[i, 2], _MASK_C, "(pas de masque)")

        # Label de ligne : nom de catégorie à gauche
        axes[i, 0].set_ylabel(
            cat, fontsize=12, fontweight="bold",
            rotation=0, ha="right", va="center", labelpad=18,
        )

    fig.suptitle(
        f"Exemples visuels — {dataset_name}",
        fontsize=15, fontweight="bold", y=0.995,
    )

    # Sous-titre / légende : bullets colorés pour chaque rôle
    fig.text(0.22, 0.975, "●", ha="center", fontsize=14, color=_NORMAL_C)
    fig.text(0.245, 0.975, "Normal", ha="left", fontsize=10,
             color=_NORMAL_C, fontweight="bold")
    fig.text(0.44, 0.975, "●", ha="center", fontsize=14, color=_ANOMAL_C)
    fig.text(0.465, 0.975, "Anomal + masque", ha="left", fontsize=10,
             color=_ANOMAL_C, fontweight="bold")
    fig.text(0.705, 0.975, "●", ha="center", fontsize=14, color=_MASK_C)
    fig.text(0.73, 0.975, "Masque seul", ha="left", fontsize=10,
             color=_MASK_C, fontweight="bold")

    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


show_samples(df, DATA.mvtec_name)
show_samples(df, DATA.hssiad_name)

## 5. Analyse des masques — Ratio de pixels anomaux

Pour chaque image anomale disposant d'un masque, on mesure le **pourcentage de pixels défectueux**.
Cela renseigne sur la **taille typique des défauts**, qui impacte fortement la difficulté de
détection et le choix des métriques (image-level vs pixel-level).

- **Gauche** : distribution globale par dataset (densité, échelle log-x, médianes en pointillés).
- **Droite** : top 10 des catégories les plus représentées, triées par médiane décroissante, avec points bruts en superposition.

In [ ]:
from matplotlib.patches import Patch

# --- 1) Compute anomaly-pixel ratio on a sample of masks -----------------
masks_df = df[df["has_mask"] == True].copy()
sample_masks = masks_df.sample(n=min(800, len(masks_df)), random_state=DATA.random_seed).copy()

ratios = []
for _, row in sample_masks.iterrows():
    try:
        mask = np.array(Image.open(PATHS.root / row["mask_path"]).convert("L"))
        ratios.append((mask > 0).sum() / mask.size * 100)
    except Exception:
        ratios.append(np.nan)

sample_masks["anomaly_pixel_pct"] = ratios
sample_masks = sample_masks.dropna(subset=["anomaly_pixel_pct"])
sample_masks = sample_masks[sample_masks["anomaly_pixel_pct"] > 0]  # log-safe

# --- 2) Palette par dataset ----------------------------------------------
palette_ds = {DATA.mvtec_name: EDA.color_mvtec, DATA.hssiad_name: EDA.color_hssiad}

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1.3]})

# --- Panel 1 : KDE par dataset (échelle log-x) ---------------------------
sns.kdeplot(
    data=sample_masks, x="anomaly_pixel_pct", hue="dataset",
    hue_order=list(palette_ds.keys()), palette=palette_ds,
    fill=True, alpha=0.35, linewidth=2,
    log_scale=(True, False), common_norm=False, ax=axes[0],
)

# Médianes + annotation
ymax = axes[0].get_ylim()[1]
for ds, color in palette_ds.items():
    sub = sample_masks[sample_masks["dataset"] == ds]
    if len(sub) == 0:
        continue
    med = sub["anomaly_pixel_pct"].median()
    axes[0].axvline(med, color=color, linestyle="--", linewidth=1.5, alpha=0.9)
    axes[0].text(
        med, ymax * 0.96, f"  médiane {med:.2f}%",
        color=color, fontsize=9, fontweight="bold",
        ha="left", va="top",
    )

axes[0].set_xlabel("% pixels anomaux (échelle log)")
axes[0].set_ylabel("Densité")
axes[0].set_title("Distribution par dataset", fontsize=12, fontweight="bold", pad=8)
if axes[0].get_legend() is not None:
    axes[0].get_legend().set_title("Dataset")
sns.despine(ax=axes[0])

# --- Panel 2 : Boxenplot top 10 catégories + stripplot -------------------
top_cats = sample_masks["category"].value_counts().head(10).index.tolist()
sub2 = sample_masks[sample_masks["category"].isin(top_cats)].copy()

cat_order = (
    sub2.groupby("category")["anomaly_pixel_pct"].median()
        .sort_values(ascending=False).index.tolist()
)
palette_cat = {
    c: palette_ds[sub2.loc[sub2["category"] == c, "dataset"].iloc[0]]
    for c in cat_order
}

sns.boxenplot(
    data=sub2, y="category", x="anomaly_pixel_pct",
    hue="category", order=cat_order, palette=palette_cat,
    linewidth=0.8, legend=False, ax=axes[1],
)
sns.stripplot(
    data=sub2, y="category", x="anomaly_pixel_pct",
    hue="category", order=cat_order, palette=palette_cat,
    size=2.8, alpha=0.45, jitter=0.25, legend=False,
    edgecolor="white", linewidth=0.2, ax=axes[1],
)

axes[1].set_xscale("log")
axes[1].set_xlabel("% pixels anomaux (échelle log)")
axes[1].set_ylabel("")
axes[1].set_title("Top 10 catégories — taille des défauts", fontsize=12, fontweight="bold", pad=8)
axes[1].grid(axis="x", linestyle=":", alpha=0.6)

# Legend manuelle (couleur = dataset de la catégorie)
axes[1].legend(
    handles=[Patch(facecolor=c, label=ds, edgecolor="white") for ds, c in palette_ds.items()],
    title="Dataset", loc="lower right", frameon=True, framealpha=0.95,
)
sns.despine(ax=axes[1])

fig.suptitle("Analyse des masques — taille relative des défauts",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# --- 3) Récap texte ------------------------------------------------------
print(f"Ratio moyen global  : {sample_masks['anomaly_pixel_pct'].mean():.2f}%")
print(f"Ratio médian global : {sample_masks['anomaly_pixel_pct'].median():.2f}%")
print("\nPar dataset :")
print(sample_masks.groupby("dataset")["anomaly_pixel_pct"].describe().round(2).to_string())

## 6. Analyse des canaux (RGB vs Grayscale)

In [ ]:
df_res["color_mode"] = df_res["channels"].map({1: "Grayscale", 3: "RGB", 4: "RGBA"}).fillna("Autre")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Par dataset
df_res.groupby(["dataset", "color_mode"]).size().unstack(fill_value=0).plot(
    kind="bar", ax=axes[0], color=["#7B7B7B", EDA.color_mvtec, EDA.color_hssiad]
)
axes[0].set_title("Canaux par dataset")
axes[0].set_ylabel("Nombre d'images (échantillon)")
axes[0].tick_params(axis='x', rotation=0)

# Par catégorie
channel_by_cat = df_res.groupby(["category", "color_mode"]).size().unstack(fill_value=0)
channel_by_cat.plot(kind="barh", stacked=True, ax=axes[1], color=["#7B7B7B", EDA.color_mvtec, EDA.color_hssiad])
axes[1].set_title("Canaux par catégorie")
axes[1].set_xlabel("Nombre d'images (échantillon)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Catégories en Grayscale :")
gray_cats = df_res[df_res["color_mode"] == "Grayscale"]["category"].unique()
print(f"  {sorted(gray_cats) if len(gray_cats) > 0 else '(aucune)'}")

## 7. Synthèse & Recommandations

In [ ]:
# Automated summary
print("=" * 60)
print("SYNTHÈSE")
print("=" * 60)

print(f"\n📊 Volume total : {len(df):,} images")
for ds in df["dataset"].unique():
    sub = df[df["dataset"] == ds]
    n_anom = sub["is_anomaly"].sum()
    n_mask = sub["has_mask"].sum()
    print(f"  {ds}: {len(sub):,} images | {sub['category'].nunique()} catégories | "
          f"{n_anom:,} anomalies | {n_mask:,} masques")

print(f"\n🔍 Couverture masques :")
total_anom = df["is_anomaly"].sum()
total_mask = df["has_mask"].sum()
missing = total_anom - total_mask
print(f"  {total_mask:,}/{total_anom:,} anomalies ont un masque ({total_mask/total_anom*100:.1f}%)")
if missing > 0:
    missing_cats = df[(df["is_anomaly"]) & (~df["has_mask"])].groupby(["dataset", "category"]).size()
    print(f"  ⚠ {missing:,} masques manquants :")
    print(missing_cats.to_string())

print(f"\n📐 Résolutions :")
for ds in df_res["dataset"].unique():
    resolutions = df_res[df_res["dataset"] == ds]["resolution"].unique()
    print(f"  {ds}: {len(resolutions)} résolution(s) uniques — {sorted(resolutions)[:5]}")

print("\n" + "=" * 60)
print("RECOMMANDATIONS POUR LE PIPELINE")
print("=" * 60)
print("""
1. RÉSOLUTION : Standardiser toutes les images à une taille commune 
   (ex: 256x256 ou 224x224) via resize + center crop.

2. CANAUX : Convertir toutes les images en RGB (les grayscale seront 
   dupliquées sur 3 canaux).

3. MASQUES MANQUANTS : Vérifier les catégories avec masques manquants. 
   Si trop de masques manquent, ces catégories ne pourront servir que 
   pour l'évaluation image-level (pas pixel-level).

4. DÉSÉQUILIBRE : Le ratio normal/anomal varie fortement entre catégories.
   Pour l'entraînement (non-supervisé), seules les images "good" du split 
   train sont utilisées — pas de problème. Pour l'évaluation, le 
   déséquilibre est normal en anomaly detection.

5. SPLIT : Garder les splits train/test originaux. Ne pas mélanger 
   pour préserver la validité des benchmarks.
""")